# Pruebas de integración — SmartLight + Keithley + Láser (Vigo)

Ejecuta las celdas **en orden**.

| Sección | Qué hace |
|---------|----------|
| 1 | Conexión SSH + medir_loop.py |
| 2 | Comprobación de polarización (por puerto) |
| 3 | Keithley solo — ejemplo de práctica |
| 4 | Integración reducida Keithley + chip |
| 5 | Barrido completo de voltaje (Keithley) |
| 6 | Medida real — 500 retos a 1550nm + barrido de longitud de onda (láser) |
| 7 | Cierre limpio |
| 8 | Guardar resultados |


## Imports comunes
Ejecutar siempre primero.

In [ ]:
import Meas_classes as ms
import paramiko
import json, time, getpass
import numpy as np
import pandas as pd

## Sección 1 — Conexión SSH + medir_loop.py
Comprueba que el SmartLight responde correctamente a un comando de prueba.

In [ ]:
PASSWORD = getpass.getpass('Contraseña SSH: ')

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect('10.42.0.125', username='smartlight', password=PASSWORD)

stdin, stdout, stderr = ssh.exec_command(
    'cd /home/smartlight/Oxel/VIGO && /home/smartlight/.venvs/ipronics/bin/python medir_loop.py'
)

def medir_remoto(inport, n_retos):
    cmd = {'inport': inport, 'n_retos': n_retos}
    stdin.write(json.dumps(cmd) + '\n')
    stdin.flush()
    return json.loads(stdout.readline())

print('✓ SSH conectado y medir_loop.py lanzado. Esperando a que el chip calibre...')
time.sleep(5)

In [ ]:
# Prueba con muy pocos retos para validar rápido
res_test = medir_remoto(inport=1, n_retos=2)
print(json.dumps(res_test, indent=2)[:1000])

Si ves un JSON con `'powers'` → **Sección 1 OK**.

Si sale `'error'`, ejecuta `print(stderr.read().decode())` en una celda nueva para ver el traceback completo.

## Sección 2 — Comprobación de polarización

**Ejecutar cada vez que cambies el puerto de entrada (INPORT).**

Flujo:
1. Mover físicamente la fibra al nuevo puerto
2. `preparar_externo(...)` → mirar el medidor analógico, ajustar la polarización a mano
3. `comprobar_polarizacion(...)` → verificar que los fotodetectores internos también ven bien la señal

In [ ]:
def preparar_externo(inport, outport_prueba):
    """Configura el chip para que el medidor analógico externo pueda leer
    la salida. Después de llamar a esto, ajusta la polarización a mano
    mirando el medidor analógico conectado físicamente al puerto."""
    cmd = {'accion': 'preparar_externo', 'inport': inport, 'outport': outport_prueba}
    stdin.write(json.dumps(cmd) + '\n')
    stdin.flush()
    res = json.loads(stdout.readline())

    if 'error' in res:
        print(f'⚠ Error: {res["error"]}')
        return None

    print(f'✓ {res["status"]}')
    return res


def comprobar_polarizacion(inport, outport_prueba):
    """Lee las potencias con los fotodetectores internos, una vez ya
    ajustada la polarización a mano con el medidor analógico."""
    cmd = {'accion': 'polarizacion', 'inport': inport, 'outport': outport_prueba}
    stdin.write(json.dumps(cmd) + '\n')
    stdin.flush()
    res = json.loads(stdout.readline())

    if 'error' in res:
        print(f'⚠ Error: {res["error"]}')
        return None

    powers = {int(k): v for k, v in res['powers'].items()}

    print(f'Entrada: puerto {inport}')
    print(f'{"Puerto salida":>14}  {"Potencia (dBm)":>15}')
    print('-' * 32)
    for puerto, potencia in sorted(powers.items()):
        barra = '█' * int((potencia + 50) / 2) if potencia > -50 else ''
        print(f'{puerto:>14}  {potencia:>15.4f}  {barra}')

    return powers

In [ ]:
# ── Cambia esto cada vez que muevas la fibra a un nuevo puerto ────────
INPORT         = 21
PRUEBA_OUTPORT = 3

preparar_externo(INPORT, PRUEBA_OUTPORT)
# → ahora mira el medidor analógico y ajusta la polarización a mano

In [ ]:
# Una vez ajustada la polarización a mano, comprobar con fotodetectores internos
comprobar_polarizacion(INPORT, PRUEBA_OUTPORT)

## Sección 3 — Keithley solo (ejemplo de práctica)
Sin tocar el SmartLight. Verifica que cambia de voltaje y lee bien.

In [ ]:
keithley = ms.Keithley('20', 'GPIB0', 0.0011, -3)
keithley.set_keithley_for_meas_current()
keithley.on_keithley_fixed_volt()

In [ ]:
VOLTAJES_TEST = [1, 3, 5]

for v in VOLTAJES_TEST:
    keithley.volt_source = v
    keithley.on_keithley_fixed_volt()
    time.sleep(0.3)
    V_medido, I_medido = keithley.read_keithley()
    print(f'V={v:.2f} V  →  V_medido={V_medido:.4f} V,  I={I_medido:.4f} mA')

keithley.off_keithley_volt()
print('\n✓ Sección 3 completada')

## Sección 4 — Integración reducida (Keithley + chip)
3 voltajes × 2 retos.

In [ ]:
keithley.on_keithley_fixed_volt()

VOLTAJES_REDUCIDO = [1, 3, 5]
INPORT_TEST = 1
N_RETOS_TEST = 2

resultados_test = []

for v in VOLTAJES_REDUCIDO:
    keithley.volt_source = v
    keithley.on_keithley_fixed_volt()
    time.sleep(0.3)

    V_medido, I_medido = keithley.read_keithley()
    res_chip = medir_remoto(inport=INPORT_TEST, n_retos=N_RETOS_TEST)

    if 'error' in res_chip:
        print(f'⚠ V={v:.2f} V → error SmartLight: {res_chip["error"]}')
        continue

    resultados_test.append({
        'voltaje_programado': v,
        'voltaje_medido':     V_medido,
        'corriente_mA':       I_medido,
        'powers':             res_chip['powers'],
    })

    print(f'V={v:.2f} V  →  V_medido={V_medido:.4f} V,  '
          f'I={I_medido:.4f} mA,  {res_chip["n_medidos"]} retos medidos')

print(f'\n✓ Sección 4 completada — {len(resultados_test)} puntos')

## Sección 5 — Barrido completo de voltaje (Keithley)
Ejecutar solo cuando las secciones 1-4 hayan funcionado correctamente.

In [ ]:
VOLTAJES    = np.linspace(1, 5, 9)   # ajusta tu rango real
INPORT_KEITHLEY = 1
N_RETOS_KEITHLEY = 5

resultados_keithley = []

keithley.on_keithley_fixed_volt()

for v in VOLTAJES:
    keithley.volt_source = v
    keithley.on_keithley_fixed_volt()
    time.sleep(0.3)

    V_medido, I_medido = keithley.read_keithley()
    res_chip = medir_remoto(inport=INPORT_KEITHLEY, n_retos=N_RETOS_KEITHLEY)

    if 'error' in res_chip:
        print(f'⚠ V={v:.2f} V → error SmartLight: {res_chip["error"]}')
        continue

    resultados_keithley.append({
        'voltaje_programado': v,
        'voltaje_medido':     V_medido,
        'corriente_mA':       I_medido,
        'powers':             res_chip['powers'],
    })

    print(f'V={v:.2f} V  →  V_medido={V_medido:.4f} V,  '
          f'I={I_medido:.4f} mA,  {res_chip["n_medidos"]} retos medidos')

## Sección 6 — Medida real: 500 retos a 1550nm + barrido de longitud de onda (láser)

**Antes de ejecutar:** completa los parámetros del láser en la celda siguiente.
Si dejas algún valor en `None`, la celda de conexión lanzará un error claro
en vez de fallar de forma confusa más adelante.

In [ ]:
# ── PARÁMETROS DEL LÁSER — RELLENAR ANTES DE CONECTAR ──────────────────
GPIB_ID_LASER      = None   # ej. '20'  (lo verás con rm.list_resources())
GPIB_ADDRESS_LASER = 'GPIB0'
POWER_LASER_DBM    = None   # ej. 0.0

WL_BASE            = 1550.0   # longitud de onda de la medida estándar (nm)
N_RETOS_BASE       = 500      # retos de la medida estándar

N_RETOS_BARRIDO    = None   # ej. 50   — retos por paso del barrido
WL_INICIO          = None   # ej. 1530.0
WL_FIN             = None   # ej. 1570.0
WL_PASO_NM         = None   # ej. 1.0

INPORT_LASER       = 21     # puerto a medir con el láser

_faltan = [nombre for nombre, val in {
    'GPIB_ID_LASER': GPIB_ID_LASER, 'POWER_LASER_DBM': POWER_LASER_DBM,
    'N_RETOS_BARRIDO': N_RETOS_BARRIDO, 'WL_INICIO': WL_INICIO,
    'WL_FIN': WL_FIN, 'WL_PASO_NM': WL_PASO_NM,
}.items() if val is None]

if _faltan:
    print(f'⚠ Faltan por rellenar: {_faltan}')
else:
    WAVELENGTHS = np.arange(WL_INICIO, WL_FIN + WL_PASO_NM, WL_PASO_NM)
    print(f'✓ Parámetros completos. Barrido: {len(WAVELENGTHS)} puntos '
          f'de {WL_INICIO} a {WL_FIN} nm cada {WL_PASO_NM} nm')

In [ ]:
# Ver qué recursos VISA detecta para localizar el láser
import pyvisa as visa
rm_temp = visa.ResourceManager()
print(rm_temp.list_resources())

In [ ]:
# ── CONECTAR LÁSER (Laser_TL17) ────────────────────────────────────────
assert GPIB_ID_LASER is not None, 'Falta GPIB_ID_LASER'
assert POWER_LASER_DBM is not None, 'Falta POWER_LASER_DBM'

laser = ms.Laser_TL17(
    gpib_id=GPIB_ID_LASER,
    gpib_address=GPIB_ADDRESS_LASER,
    power=POWER_LASER_DBM,
    wl=WL_BASE,
)
laser.set_power_laser()
laser.set_wl_laser()
laser.on_laser()
time.sleep(1.0)
print(f'✓ Láser encendido a {WL_BASE} nm, {POWER_LASER_DBM} dBm')

In [ ]:
# ── RECORRIDO AUTOMÁTICO POR PUERTOS ───────────────────────────────────
import glob, re

PUERTOS_A_MEDIR = ACTIVE_PORTS   # o una lista concreta, ej. [21, 22, 23]

# Repoblar PUERTOS_HECHOS mirando qué archivos ya existen en disco
# (útil si reinicias el kernel a mitad de sesión)
PUERTOS_HECHOS = []
for fname in glob.glob('powers_base_1550nm_port*.json'):
    m = re.search(r'port(\d+)\.json$', fname)
    if m:
        PUERTOS_HECHOS.append(int(m.group(1)))
PUERTOS_HECHOS = sorted(set(PUERTOS_HECHOS))

if PUERTOS_HECHOS:
    print(f'✓ Puertos ya medidos (detectados en disco): {PUERTOS_HECHOS}')
else:
    print('No hay puertos medidos todavía — empezando de cero')

for puerto in PUERTOS_A_MEDIR:
    if puerto in PUERTOS_HECHOS:
        continue

    print(f'\n{"="*50}\nPUERTO {puerto}\n{"="*50}')
    input(f'Conecta la fibra al puerto {puerto} y pulsa Enter...')

    # Puerto de salida para polarización — normalmente el mismo, pero
    # cambiable puerto a puerto si hace falta (Enter = mantener el de antes)
    _outport_str = input(f'Puerto de salida para polarización (Enter = {PRUEBA_OUTPORT}): ').strip()
    outport_actual = int(_outport_str) if _outport_str else PRUEBA_OUTPORT

    # 1. Preparar modo externo y ajustar polarización a mano
    preparar_externo(puerto, outport_actual)
    input('Ajusta la polarización con el medidor analógico y pulsa Enter...')

    # 2. Comprobar con fotodetectores internos
    comprobar_polarizacion(puerto, outport_actual)
    continuar = input('¿Polarización OK? (Enter para continuar, "r" para repetir el puerto) ')
    if continuar.strip().lower() == 'r':
        continue   # repite el mismo puerto sin marcarlo como hecho

    # 3. Medida estándar a 1550nm (500 retos)
    print(f'Midiendo {N_RETOS_BASE} retos a {WL_BASE} nm...')
    res_base = medir_remoto(inport=puerto, n_retos=N_RETOS_BASE)

    if 'error' in res_base:
        print(f'⚠ Error en medida base puerto {puerto}: {res_base["error"]}')
        continue   # no marca el puerto como hecho, lo reintentará otro día

    fname_base = f'powers_base_1550nm_port{puerto:02d}.json'
    with open(fname_base, 'w') as f:
        json.dump(res_base, f, indent=2)
    print(f'✓ Medida base guardada — {res_base["n_medidos"]} retos → {fname_base}')

    # 4. Barrido de longitud de onda
    resultados_barrido = []
    for wl in WAVELENGTHS:
        laser.wl = float(wl)
        laser.set_wl_laser()
        time.sleep(0.5)

        res = medir_remoto(inport=puerto, n_retos=N_RETOS_BARRIDO)
        if 'error' in res:
            print(f'⚠ λ={wl:.2f} nm → {res["error"]}')
            continue

        resultados_barrido.append({'wavelength': float(wl), 'powers': res['powers']})
        print(f'λ={wl:.2f} nm  →  {res["n_medidos"]} retos medidos')

    fname_barrido = f'barrido_lambda_port{puerto:02d}.json'
    with open(fname_barrido, 'w') as f:
        json.dump(resultados_barrido, f, indent=2)
    print(f'✓ {len(resultados_barrido)} puntos de λ guardados → {fname_barrido}')

    # 5. Marcar puerto como hecho
    PUERTOS_HECHOS.append(puerto)
    print(f'\n✓✓ Puerto {puerto} completado ({len(PUERTOS_HECHOS)}/{len(PUERTOS_A_MEDIR)})')

print(f'\n{"="*50}\nSESIÓN COMPLETA — {len(PUERTOS_HECHOS)} puertos medidos\n{"="*50}')
print(f'Puertos hechos:      {PUERTOS_HECHOS}')
print(f'Puertos pendientes:  {[p for p in PUERTOS_A_MEDIR if p not in PUERTOS_HECHOS]}')

## Sección 7 — Cierre limpio
Ejecutar **siempre al terminar**, aunque haya habido errores.

In [ ]:
try:
    keithley.off_keithley_volt()
    print('✓ Keithley apagado')
except Exception as e:
    print(f'Keithley ya apagado o no conectado: {e}')

try:
    laser.off_laser()
    print('✓ Láser apagado')
except Exception as e:
    print(f'Láser ya apagado o no conectado: {e}')

try:
    stdin.write('EXIT\n')
    stdin.flush()
    ssh.close()
    print('✓ SmartLight desconectado, SSH cerrado')
except Exception as e:
    print(f'SSH ya cerrado: {e}')

## Sección 8 — Guardar resultados
Ejecutar tras la Sección 5 (Keithley) y/o Sección 6 (láser).

In [ ]:
# Resultados del barrido de voltaje (Keithley)
if 'resultados_keithley' in dir() and resultados_keithley:
    with open('barrido_voltaje_chip.json', 'w') as f:
        json.dump(resultados_keithley, f, indent=2)

    df_resumen = pd.DataFrame([
        {'voltaje_programado': r['voltaje_programado'],
         'voltaje_medido':     r['voltaje_medido'],
         'corriente_mA':       r['corriente_mA'],
         'n_retos':            len(r['powers'])}
        for r in resultados_keithley
    ])
    df_resumen.to_csv('barrido_voltaje_resumen.csv', index=False)
    print(f'✓ {len(resultados_keithley)} puntos de voltaje guardados')
    print(df_resumen)

In [ ]:
# Resultados del barrido de longitud de onda (láser)
if 'resultados_barrido' in dir() and resultados_barrido:
    with open(f'barrido_lambda_port{INPORT_LASER:02d}.json', 'w') as f:
        json.dump(resultados_barrido, f, indent=2)
    print(f'✓ {len(resultados_barrido)} puntos de λ guardados')